# Exploring Rotations and Spatial Orientation

What's the difference between rotating the whole cube and turning a single face? Both change what the cube looks like, but they represent fundamentally different operations. A **face move** (like `R`) permutes pieces — corners and edges swap positions. A **rotation** (like `x`) changes your **viewpoint** — you're looking at the same puzzle from a different angle.

This distinction, invisible at the facelet level where everything just shuffles around, becomes crystal clear in the cubie representation through the **SO (Spatial Orientation) array** — a hidden sixth dimension of cube state that tracks which original face occupies each position.

This notebook goes deep into this duality and explores five rotation-related transform modules:

| Module | Purpose |
|--------|---------|
| **offset** | Rename face moves as if you rotated your viewpoint |
| **rotation** | Optimize and compress rotation sequences |
| **degrip** | Absorb mid-algorithm rotations into face moves |
| **symmetry** | Mirror algorithms across slice planes |
| **translate** | Adapt algorithms for a different point-of-view orientation |

### Prerequisites
- Familiarity with the `Algorithm` and `VCube` classes (notebooks 02 and 03)
- Understanding of the three cube representations (notebook 07)
- Basic awareness of algorithm transforms (notebook 05)

In [1]:
from cubing_algs import Algorithm
from cubing_algs import VCube
from cubing_algs.constants import CORNER_NAMES
from cubing_algs.constants import EDGE_NAMES
from cubing_algs.constants import FACE_ORDER
from cubing_algs.constants import OFFSET_ORIENTATION_MAP
from cubing_algs.constants import OFFSET_TABLE
from cubing_algs.constants import ORIENTATIONS
from cubing_algs.constants import SOLVED_SO
from cubing_algs.constants import SYMMETRY_TABLE
from cubing_algs.transform.degrip import DEGRIP_FULL
from cubing_algs.transform.degrip import degrip_full_moves
from cubing_algs.transform.degrip import degrip_x_moves
from cubing_algs.transform.degrip import has_grip
from cubing_algs.transform.offset import offset_moves
from cubing_algs.transform.offset import offset_x_moves
from cubing_algs.transform.offset import offset_y_moves
from cubing_algs.transform.offset import offset_z_moves

# Transform modules — the five we'll explore in depth
from cubing_algs.transform.offset import rotate
from cubing_algs.transform.rotation import compress_ending_rotations
from cubing_algs.transform.rotation import compress_rotations
from cubing_algs.transform.rotation import remove_ending_rotations
from cubing_algs.transform.rotation import remove_rotations
from cubing_algs.transform.rotation import remove_starting_rotations
from cubing_algs.transform.rotation import split_moves_ending_rotations
from cubing_algs.transform.size import compress_moves
from cubing_algs.transform.symmetry import symmetry_c_moves
from cubing_algs.transform.symmetry import symmetry_e_moves
from cubing_algs.transform.symmetry import symmetry_m_moves
from cubing_algs.transform.symmetry import symmetry_s_moves
from cubing_algs.transform.translate import translate_moves
from cubing_algs.transform.translate import translate_pov_moves
from cubing_algs.transform.wide import rewide_moves
from cubing_algs.transform.wide import unwide_rotation_moves


def show_cubies(cube: VCube) -> None:
    """Display cubie arrays with piece names for readability."""
    cp, co, ep, eo, so = cube.cubies
    print(f"  CP: {cp}  ({', '.join(CORNER_NAMES[c] for c in cp)})")
    print(f'  CO: {co}')
    print(f'  EP: {ep}')
    print(f"      ({', '.join(EDGE_NAMES[e] for e in ep)})")
    print(f'  EO: {eo}')
    print(f"  SO: {so}  ({', '.join(FACE_ORDER[s] for s in so)})")


# Reference algorithms used throughout
SEXY_MOVE = Algorithm.parse_moves("R U R' U'")
T_PERM = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")

print('=== Setup Complete ===')
print(f'SOLVED_SO = {SOLVED_SO}  (maps to {FACE_ORDER})')
print('24 valid orientations available')
print(f'Reference: Sexy move = {SEXY_MOVE}, T-Perm = {T_PERM}')

=== Setup Complete ===
SOLVED_SO = [0, 1, 2, 3, 4, 5]  (maps to ('U', 'R', 'F', 'D', 'L', 'B'))
24 valid orientations available
Reference: Sexy move = R U R' U', T-Perm = R U R' F' R U R' U' R' F R2 U' R'


## What Happens When You Rotate the Cube

Pick up a solved Rubik's cube and turn the R face. A few pieces move — the right-side corners and edges swap around. Now instead, rotate the entire cube forward (an `x` rotation). Every single sticker appears to move, yet no piece has actually changed position relative to any other piece. You've just changed your viewpoint.

At the **facelet level**, both operations look similar — stickers shuffle around in the 54-character string. But the underlying reality is completely different:

- **Face move (`R`)**: Pieces physically change positions. Some corners and edges move to new slots.
- **Rotation (`x`)**: Your frame of reference changes. What was the U face is now the F face, what was F is now D, etc.

This is the **key duality** that drives everything in this notebook: face moves permute pieces, rotations change your viewpoint.

In [2]:
print('=== Rotation vs Face Move — Facelet Level ===')
print()

solved = VCube()
solved_state = solved.state

# Apply an x rotation
cube_rotated = VCube()
cube_rotated.rotate('x')

# Apply an R face move
cube_face = VCube()
cube_face.rotate('R')

# Count changed facelets
rotation_changes = sum(1 for i in range(54) if solved_state[i] != cube_rotated.state[i])
face_changes = sum(1 for i in range(54) if solved_state[i] != cube_face.state[i])

print('--- After x rotation ---')
print(f'  State: {cube_rotated.state}')
print(f'  Changed facelets: {rotation_changes}/54')

print('\n--- After R face move ---')
print(f'  State: {cube_face.state}')
print(f'  Changed facelets: {face_changes}/54')

# Show which faces moved entirely vs partially
print('\n--- Face-by-face breakdown ---')
print(f"{'Face':<6} {'x rotation':<20} {'R face move':<20}")
print('-' * 46)
for i, face in enumerate(FACE_ORDER):
    s = i * 9
    rot_face = cube_rotated.state[s:s + 9]
    r_face = cube_face.state[s:s + 9]
    rot_changed = sum(1 for j in range(9) if solved_state[s + j] != rot_face[j])
    r_changed = sum(1 for j in range(9) if solved_state[s + j] != r_face[j])
    print(f'  {face:<4}   {rot_face} ({rot_changed} changed)   {r_face} ({r_changed} changed)')

print(f'\nKey insight: x rotation moves ENTIRE faces (all {rotation_changes} facelets),')
print(f'while R only disturbs {face_changes} facelets across a few faces.')

=== Rotation vs Face Move — Facelet Level ===

--- After x rotation ---
  State: FFFFFFFFFRRRRRRRRRDDDDDDDDDBBBBBBBBBLLLLLLLLLUUUUUUUUU
  Changed facelets: 36/54

--- After R face move ---
  State: UUFUUFUUFRRRRRRRRRFFDFFDFFDDDBDDBDDBLLLLLLLLLUBBUBBUBB
  Changed facelets: 12/54

--- Face-by-face breakdown ---
Face   x rotation           R face move         
----------------------------------------------
  U      FFFFFFFFF (9 changed)   UUFUUFUUF (3 changed)
  R      RRRRRRRRR (0 changed)   RRRRRRRRR (0 changed)
  F      DDDDDDDDD (9 changed)   FFDFFDFFD (3 changed)
  D      BBBBBBBBB (9 changed)   DDBDDBDDB (3 changed)
  L      LLLLLLLLL (0 changed)   LLLLLLLLL (0 changed)
  B      UUUUUUUUU (9 changed)   UBBUBBUBB (3 changed)

Key insight: x rotation moves ENTIRE faces (all 36 facelets),
while R only disturbs 12 facelets across a few faces.


## The Cubie Perspective — Where the Magic Happens

The facelet view treats rotations and face moves as "the same kind of thing" — both shuffle stickers. The cubie representation reveals they are fundamentally different operations:

- **Face moves** change `cp/co/ep/eo` (piece positions and orientations) but leave `so` untouched
- **Rotations** change `so` (spatial orientation) but leave `cp/co/ep/eo` untouched

The **SO (Spatial Orientation) array** has 6 elements: `so[i]` tells you which original face now occupies position `i`. The positions follow `FACE_ORDER = [U, R, F, D, L, B]`.

For a solved cube: `SOLVED_SO = [0, 1, 2, 3, 4, 5]` — face 0 (U) is at position 0 (top), face 1 (R) is at position 1 (right), etc.

After an `x` rotation (tilt forward): U goes to B position, F goes to U, D goes to F, B goes to D — but the pieces themselves haven't moved relative to each other.

In [3]:
print('=== Rotation vs Face Move — Cubie Level ===')
print()

# Solved state baseline
solved = VCube()
s_cp, s_co, s_ep, s_eo, s_so = solved.cubies

# After R face move
print('--- After R (face move) ---')
cube_r = VCube()
cube_r.rotate('R')
r_cp, r_co, r_ep, r_eo, r_so = cube_r.cubies
print(f'  CP changed: {r_cp != s_cp}  {r_cp}')
print(f'  CO changed: {r_co != s_co}  {r_co}')
print(f'  EP changed: {r_ep != s_ep}  {r_ep}')
print(f'  EO changed: {r_eo != s_eo}  {r_eo}')
print(f'  SO changed: {r_so != s_so}  {r_so}  <-- UNCHANGED!')

# After x rotation
print('\n--- After x (rotation) ---')
cube_x = VCube()
cube_x.rotate('x')
x_cp, x_co, x_ep, x_eo, x_so = cube_x.cubies
print(f'  CP changed: {x_cp != s_cp}  {x_cp}  <-- UNCHANGED!')
print(f'  CO changed: {x_co != s_co}  {x_co}  <-- UNCHANGED!')
print(f'  EP changed: {x_ep != s_ep}  {x_ep}  <-- UNCHANGED!')
print(f'  EO changed: {x_eo != s_eo}  {x_eo}  <-- UNCHANGED!')
print(f"  SO changed: {x_so != s_so}  {x_so}  ({', '.join(FACE_ORDER[s] for s in x_so)})")

# After R x (both)
print('\n--- After R x (face move + rotation) ---')
cube_rx = VCube()
cube_rx.rotate('R x')
rx_cp, rx_co, rx_ep, rx_eo, rx_so = cube_rx.cubies
print(f'  CP changed: {rx_cp != s_cp}  (pieces permuted by R)')
print(f'  SO changed: {rx_so != s_so}  (viewpoint changed by x)')

# Summary table
print('\n--- Summary ---')
print(f"{'Move':<8} {'cp/co/ep/eo':<20} {'so':<20}")
print('-' * 48)
print(f"{'R':<8} {'CHANGED':<20} {'unchanged':<20}")
print(f"{'x':<8} {'unchanged':<20} {'CHANGED':<20}")
print(f"{'R x':<8} {'CHANGED (by R)':<20} {'CHANGED (by x)':<20}")

=== Rotation vs Face Move — Cubie Level ===

--- After R (face move) ---
  CP changed: True  [4, 1, 2, 0, 7, 5, 6, 3]
  CO changed: True  [2, 0, 0, 1, 1, 0, 0, 2]
  EP changed: True  [8, 1, 2, 3, 11, 5, 6, 7, 4, 9, 10, 0]
  EO changed: False  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  SO changed: False  [0, 1, 2, 3, 4, 5]  <-- UNCHANGED!

--- After x (rotation) ---
  CP changed: False  [0, 1, 2, 3, 4, 5, 6, 7]  <-- UNCHANGED!
  CO changed: False  [0, 0, 0, 0, 0, 0, 0, 0]  <-- UNCHANGED!
  EP changed: False  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  <-- UNCHANGED!
  EO changed: False  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  <-- UNCHANGED!
  SO changed: True  [2, 1, 3, 5, 4, 0]  (F, R, D, B, L, U)

--- After R x (face move + rotation) ---
  CP changed: True  (pieces permuted by R)
  SO changed: True  (viewpoint changed by x)

--- Summary ---
Move     cp/co/ep/eo          so                  
------------------------------------------------
R        CHANGED              unchanged           
x    

## The 24 Orientations

A cube has exactly **24 valid orientations** — the rotational symmetries of a cube. Each is uniquely identified by which face is on **top** and which is in **front**:

- 6 choices for the top face
- 4 choices for the front face (any adjacent face, not opposite)
- 6 x 4 = 24 orientations

The `ORIENTATIONS` constant lists all 24 as two-character codes (e.g., `'UF'` = U on top, F in front).

Key VCube methods:
- `cube.orientation` — reads the center facelets to determine current orientation
- `cube.compute_orientation_moves(faces)` — finds the rotation sequence to reach a target orientation
- `cube.is_equal(other, strict=True/False)` — strict checks exact facelets; non-strict ignores orientation

In [4]:
print('=== The 24 Orientations ===')
print()

# List all 24 orientations grouped by top face
print('--- All 24 Valid Orientations ---')
for top_face in FACE_ORDER:
    group = [o for o in ORIENTATIONS if o[0] == top_face]
    print(f'  Top={top_face}: {group}')

# Current orientation of solved cube
cube = VCube()
print('\n--- Orientation of a Solved Cube ---')
print(f'  Orientation: {cube.orientation}  (U on top, F in front)')

# Rotate and check orientation
print('\n--- Rotating to Different Orientations ---')
rotations = ['x', 'y', 'z', 'x2', 'x y']
for rot in rotations:
    c = VCube()
    c.rotate(rot)
    print(f'  After {rot:5s}: orientation = {c.orientation}')

# compute_orientation_moves: find path to target
print('\n--- Finding Rotation Paths ---')
cube = VCube()
targets = ['RF', 'DB', 'FU', 'LB']
for target in targets:
    moves = cube.compute_orientation_moves(target)
    print(f"  UF -> {target}: {moves or '(already there)'}")

# is_equal with strict vs non-strict
print('\n--- Orientation and Equality ---')
cube_uf = VCube()
cube_uf.rotate("R U R' U'")

cube_rf = VCube()
cube_rf.rotate("z'")
cube_rf.rotate("R U R' U'")  # same scramble but from RF orientation

# Wait — that doesn't apply the same scramble. Let's use oriented_copy:
cube_reoriented = cube_uf.oriented_copy('RF')

print(f'  cube_uf orientation:       {cube_uf.orientation}')
print(f'  cube_reoriented orient.:   {cube_reoriented.orientation}')
print(f'  Same facelets (strict)?    {cube_uf.is_equal(cube_reoriented, strict=True)}')
print(f'  Same puzzle (non-strict)?  {cube_uf.is_equal(cube_reoriented, strict=False)}')

=== The 24 Orientations ===

--- All 24 Valid Orientations ---
  Top=U: ['UF', 'UR', 'UL', 'UB']
  Top=R: ['RF', 'RD', 'RB', 'RU']
  Top=F: ['FD', 'FR', 'FU', 'FL']
  Top=D: ['DF', 'DR', 'DL', 'DB']
  Top=L: ['LF', 'LD', 'LB', 'LU']
  Top=B: ['BD', 'BR', 'BU', 'BL']

--- Orientation of a Solved Cube ---
  Orientation: UF  (U on top, F in front)

--- Rotating to Different Orientations ---
  After x    : orientation = FD
  After y    : orientation = UR
  After z    : orientation = LF
  After x2   : orientation = DB
  After x y  : orientation = FR

--- Finding Rotation Paths ---
  UF -> RF: z'
  UF -> DB: x2
  UF -> FU: y2 x'
  UF -> LB: y2 z'

--- Orientation and Equality ---
  cube_uf orientation:       UF
  cube_reoriented orient.:   RF
  Same facelets (strict)?    False
  Same puzzle (non-strict)?  True


## Orientation Equivalence — Same Puzzle, Different Viewpoint

Cubes at different orientations represent the **same puzzle state viewed from different angles**. `oriented_copy(faces)` creates a new VCube reoriented to specified faces by applying rotation moves internally.

Since `oriented_copy` physically rotates the facelets, the cubie arrays (`cp/co/ep/eo`) are recomputed relative to the new center positions — so they look different in each orientation. But `is_equal(strict=False)` sees through this by normalizing both cubes to the same orientation before comparing.

- The Kociemba solver requires `UF` orientation — this is why `oriented_copy('UF')` is used before solving

**When orientation matters**: display, communication, visual pattern matching

**When it doesn't**: algorithm analysis, solving, classification (PLL/OLL), cycle counting

In [5]:
print('=== Orientation Equivalence ===')
print()

# Scramble a cube
original = VCube()
original.rotate("R U R' F' R U R' U' R' F R2 U' R'")

# Create oriented copies at different orientations
orientations_to_test = ['UF', 'RF', 'DB', 'FU']
copies = {o: original.oriented_copy(o) for o in orientations_to_test}

print('--- Same Puzzle, Different Viewpoints ---')
for orient, cube in copies.items():
    cp, co, ep, eo, so = cube.cubies
    print(f'  {orient}: facelets={cube.state[:18]}...  SO={so}')

# Show that cubie arrays differ across orientations
# (because they're computed relative to the current center positions)
print('\n--- Cubie Arrays Differ by Reference Frame ---')
ref_cp, ref_co, ref_ep, ref_eo, _ = copies['UF'].cubies
for orient, cube in copies.items():
    cp, co, ep, eo, _ = cube.cubies
    same = (cp == ref_cp and co == ref_co and ep == ref_ep and eo == ref_eo)
    print(f'  {orient}: cp/co/ep/eo same as UF? {same}')
print('  (Cubie arrays are relative to centers — different orientation, different numbers)')

# is_equal for all pairs — non-strict normalizes before comparing
print('\n--- Equality Checks ---')
orient_list = list(copies.keys())
for i in range(len(orient_list)):
    for j in range(i + 1, len(orient_list)):
        a, b = orient_list[i], orient_list[j]
        strict = copies[a].is_equal(copies[b], strict=True)
        loose = copies[a].is_equal(copies[b], strict=False)
        print(f'  {a} vs {b}: strict={strict}, non-strict={loose}')

# Preparing for solver
print('\n--- Solver Preparation ---')
messy = VCube()
messy.rotate("z R U R' U'")  # scramble with initial rotation
print(f'  Orientation after z scramble: {messy.orientation}')
ready = messy.oriented_copy('UF')
print(f"  After oriented_copy('UF'):    {ready.orientation}")
print(f'  Same puzzle? {messy.is_equal(ready, strict=False)}')

=== Orientation Equivalence ===

--- Same Puzzle, Different Viewpoints ---
  UF: facelets=UUUUUUUUURBBRRRRRR...  SO=[0, 1, 2, 3, 4, 5]
  RF: facelets=BRRBRRRRRDDDDDDDDD...  SO=[1, 3, 2, 4, 0, 5]
  DB: facelets=DDDDDDDDDRRRRRRBBR...  SO=[3, 1, 5, 0, 4, 2]
  FU: facelets=FFFFFFFFBRLLRLLFLL...  SO=[2, 4, 0, 5, 1, 3]

--- Cubie Arrays Differ by Reference Frame ---
  UF: cp/co/ep/eo same as UF? True
  RF: cp/co/ep/eo same as UF? False
  DB: cp/co/ep/eo same as UF? False
  FU: cp/co/ep/eo same as UF? False
  (Cubie arrays are relative to centers — different orientation, different numbers)

--- Equality Checks ---
  UF vs RF: strict=False, non-strict=True
  UF vs DB: strict=False, non-strict=True
  UF vs FU: strict=False, non-strict=True
  RF vs DB: strict=False, non-strict=True
  RF vs FU: strict=False, non-strict=True
  DB vs FU: strict=False, non-strict=True

--- Solver Preparation ---
  Orientation after z scramble: LF
  After oriented_copy('UF'):    UF
  Same puzzle? True


## Transform Deep Dive 1 — Offset (Viewpoint Rotation)

**Module**: `cubing_algs.transform.offset`

Offset is the core building block for rotation-aware transforms. Given a single rotation (x, y, z, or their primes), `rotate` rewrites every move in an algorithm so it targets the same physical pieces when viewed from the rotated perspective. It is a pure remapping — no moves are added or removed.

**Example** — offset by y (the cube turns clockwise from above, like U):

After y the cube looks like this:

| position | U | R | F | D | L | B |
|----------|---|---|---|---|---|---|
| stickers | W | B | R | Y | G | O |

The R face (red) is now in front. From the y-shifted viewpoint the user calls it "F". Every face that moved gets a new name: `R → F`, `F → L`, `L → B`, `B → R` (U and D stay).

```
offset_y_moves( R  U  R' U' )
              → F  U  F' U'
```

**Naming convention — perspective shift, not rotation applied:**

`offset_y_moves` means "rewrite for a y-shifted viewpoint." Internally it applies the *inverse* rotation table (y') to each move. This is why the function name and the rotation string look swapped:

| Function | Internal call | Table used |
|----------|--------------|------------|
| `offset_y_moves` | `offset_moves(algo, "y'")` | y' table |
| `offset_yprime_moves` | `offset_moves(algo, "y")` | y table |

The inverse is needed because we are translating into the rotated frame: "what does the original R become if the observer has turned by y?" — the R face is now in front, so it becomes F.

Key functions:
- `rotate(algo, rotation)` — apply a single rotation remapping
- `offset_x_moves(algo)` — shorthand for x' offset (viewpoint tilt forward)
- `offset_moves(algo, rotation, count)` — apply rotation multiple times

The `OFFSET_TABLE` maps each rotation to its face remapping dictionary. For example, `OFFSET_TABLE['x']` maps `U→F, F→D, D→B, B→U` (and similarly for slices and other rotations).

**Relationship with other transform modules:**

- **offset** applies a single, known rotation to every move — pure remapping.
- **degrip** scans for inline rotations, removes each one, and uses offset to rewrite the moves that follow.
- **translate** handles a full orientation (possibly multi-rotation, e.g. z2 or x y) applied to the whole algorithm at once, chaining offset calls for each rotation.
- **translate_pov** walks left-to-right accumulating rotations, translating only non-rotation moves that follow each rotation into the user's current point of view.

In [6]:
print('=== Offset Transform ===')
print()

# Show the OFFSET_TABLE for x rotation
print("--- OFFSET_TABLE['x'] (face remapping) ---")
for old, new in OFFSET_TABLE['x'].items():
    print(f'  {old} -> {new}')

# Apply offset to a familiar algorithm
alg = Algorithm.parse_moves("R U R' U'")
print('\n--- Offset Examples ---')
print(f'  Original:    {alg}')
print(f'  offset_x:    {alg.transform(offset_x_moves)}')
print(f'  offset_y:    {alg.transform(offset_y_moves)}')
print(f'  offset_z:    {alg.transform(offset_z_moves)}')

# Using rotate() directly
xprime = "x'"
print(f"\n  rotate(x):   {rotate(alg, 'x')}")
print(f"  rotate(x'):  {rotate(alg, xprime)}")

# Double offset
print(f"\n  offset y x2: {offset_moves(alg, 'y', 2)}")

# Verification: offset enables rotation absorption
# "x' R U R' U'" should produce the same state as "offset_x(R U R' U') x'"
# because offset_x moves the x' rotation from before to after the face moves
print('\n--- Verification: Offset Absorbs Rotations ---')
original_alg = Algorithm.parse_moves("R U R' U'")
offset_alg = original_alg.transform(offset_x_moves)

cube1 = VCube()
cube1.rotate("x'")
cube1.rotate(original_alg)  # x' then R U R' U'

cube2 = VCube()
cube2.rotate(offset_alg)
cube2.rotate("x'")  # offset(R U R' U') then x'

print(f"  x' then R U R' U':             {cube1.state}")
print(f"  {offset_alg} then x':             {cube2.state}")
print(f'  Identical?                     {cube1.state == cube2.state}')
print('  This is how degrip works — rotations move to the end!')

# Show OFFSET_ORIENTATION_MAP (a few entries)
print('\n--- OFFSET_ORIENTATION_MAP (sample) ---')
print('  Maps orientation codes to rotation sequences:')
for key in ['0', '1', '2', '3', '4', '5']:
    orient_name = FACE_ORDER[int(key)] if key.isdigit() and int(key) < 6 else key
    print(f"  '{key}' ({orient_name} on top): '{OFFSET_ORIENTATION_MAP[key]}'")

=== Offset Transform ===

--- OFFSET_TABLE['x'] (face remapping) ---
  U -> F
  D -> B
  F -> D
  B -> U
  S -> E
  E -> S'
  y -> z
  z -> y'

--- Offset Examples ---
  Original:    R U R' U'
  offset_x:    R B R' B'
  offset_y:    F U F' U'
  offset_z:    D R D' R'

  rotate(x):   R F R' F'
  rotate(x'):  R B R' B'

  offset y x2: L U L' U'

--- Verification: Offset Absorbs Rotations ---
  x' then R U R' U':             BBLBBUBBURRBDRRBRRUUFUUBUUUFFRFFFFFFDLLLLLLLLDRRDDDDDD
  R B R' B' then x':             BBLBBUBBURRBDRRBRRUUFUUBUUUFFRFFFFFFDLLLLLLLLDRRDDDDDD
  Identical?                     True
  This is how degrip works — rotations move to the end!

--- OFFSET_ORIENTATION_MAP (sample) ---
  Maps orientation codes to rotation sequences:
  '0' (U on top): ''
  '1' (R on top): 'z''
  '2' (F on top): 'x'
  '3' (D on top): 'z2'
  '4' (L on top): 'z'
  '5' (B on top): 'x''


## Transform Deep Dive 2 — Rotation Management

**Module**: `cubing_algs.transform.rotation`

Rotation sequences often contain redundancies. This module provides functions to compress, strip, and split rotation moves:

| Function | Purpose | Example |
|----------|---------|---------|
| `compress_rotations` | Simulates rotations on a VCube, reads the resulting orientation, and returns the shortest equivalent sequence (at most 2 moves) | `x y x' y' x2 y2 z2` → (empty) |
| `compress_ending_rotations` | Separates core moves from trailing rotations, compresses only the trailing part | `R U R' x y x` → `R U R' z'` |
| `remove_rotations` | Strip all rotation moves | Keeps only face moves |
| `remove_starting_rotations` | Strip leading rotations and pauses | Clean algorithm start |
| `remove_ending_rotations` | Strip trailing rotations and pauses | Clean algorithm end |
| `split_moves_ending_rotations` | Separate trailing rotations from core | Returns `(core, rotations)` tuple |

`compress_rotations` is the workhorse — instead of pattern-matching specific redundancies (like consecutive same-axis moves or conjugate pairs), it takes a fundamentally different approach: simulate the full rotation sequence on a virtual cube, read the resulting orientation, and look up the shortest rotation sequence that produces that orientation. This guarantees optimal compression in a single pass, regardless of how tangled the input rotations are.

In [7]:
print('=== Rotation Management ===')
print()

# compress_rotations: simulate and find optimal form
print('--- compress_rotations (VCube simulation approach) ---')
examples = [
    ('x x', 'two x = x2'),
    ('x x x', "three x = x'"),
    ("x x'", "x and x' cancel"),
    ('y y y y', 'four y = identity'),
    ('x2 y2 z2', 'triple double = identity'),
    ('x2 y2', 'two doubles = one double'),
    ("y x2 y'", 'conjugate simplification'),
]
for alg_str, desc in examples:
    alg = Algorithm.parse_moves(alg_str)
    result = alg.transform(compress_rotations)
    print(f"  {alg!s:<14} -> {str(result) if result else '(empty)':<8} ({desc})")

# compress_ending_rotations: only compress the trailing part
print('\n--- compress_ending_rotations ---')
trailing_examples = [
    "R U R' x y x",
    "R U R' U' x x'",
    "R U R' y y y",
]
for alg_str in trailing_examples:
    alg = Algorithm.parse_moves(alg_str)
    result = alg.transform(compress_ending_rotations)
    print(f'  {alg!s:<22} -> {result}')

# split_moves_ending_rotations
print('\n--- Splitting Core Moves from Trailing Rotations ---')
alg = Algorithm.parse_moves("R U R' U' x y2")
core, rotations = split_moves_ending_rotations(alg)
print(f'  Full: {alg}')
print(f'  Core: {core}')
print(f'  Tail: {rotations}')

# remove_rotations / remove_starting / remove_ending
print('\n--- Removing Rotations ---')
alg = Algorithm.parse_moves("x R U R' U' y z")
print(f'  Original:           {alg}')
print(f'  remove_rotations:   {alg.transform(remove_rotations)}')
print(f'  remove_starting:    {alg.transform(remove_starting_rotations)}')
print(f'  remove_ending:      {alg.transform(remove_ending_rotations)}')

# Full pipeline demo
print('\n--- Full Compression Pipeline ---')
messy = Algorithm.parse_moves("x y x' y' x2 y2 z2 R U R' z z'")
print(f'  Before: {messy}')
# First compress the rotation-only prefix, then handle the rest
clean = messy.transform(degrip_full_moves, compress_ending_rotations)
print(f'  After degrip + compress ending: {clean}')

=== Rotation Management ===

--- compress_rotations (VCube simulation approach) ---
  x x            -> x2       (two x = x2)
  x x x          -> x'       (three x = x')
  x x'           -> (empty)  (x and x' cancel)
  y y y y        -> (empty)  (four y = identity)
  x2 y2 z2       -> (empty)  (triple double = identity)
  x2 y2          -> z2       (two doubles = one double)
  y x2 y'        -> z2       (conjugate simplification)

--- compress_ending_rotations ---
  R U R' x y x           -> R U R' y2 z
  R U R' U' x x'         -> R U R' U'
  R U R' y y y           -> R U R' y'

--- Splitting Core Moves from Trailing Rotations ---
  Full: R U R' U' x y2
  Core: R U R' U'
  Tail: x y2

--- Removing Rotations ---
  Original:           x R U R' U' y z
  remove_rotations:   R U R' U'
  remove_starting:    R U R' U' y z
  remove_ending:      x R U R' U'

--- Full Compression Pipeline ---
  Before: x y x' y' x2 y2 z2 R U R' z z'
  After degrip + compress ending: F L F' y' x


## Transform Deep Dive 3 — Degrip (Absorbing Rotations)

**Module**: `cubing_algs.transform.degrip`

A "grip" is a cube rotation (x, y, z) that appears before non-rotation moves. Mid-algorithm rotations are a speedcuber's nemesis — they require physically regripping the cube, breaking flow and costing time. **Degrip** solves this by absorbing rotation moves into the subsequent face moves using the inverse offset.

**How it works:** the algorithm scans left-to-right for the first rotation listed in the config. When found, it applies the inverse offset to every move that follows, effectively absorbing the rotation. The rotation itself is pushed to the end of the algorithm (where it becomes a trailing rotation that can later be stripped by `remove_ending_rotations`).

If the result still contains grips, the process repeats until none remain.

**Example:**

```
Input:   x  R  U  R' U'
         ^  ^^^^^^^^^^^
         grip  face moves (absolute frame)

Step 1:  apply x' offset to  R U R' U'  →  R F R' F'
Result:  R  F  R' F'  x
         ^^^^^^^^^^^^  ^
         degripped      trailing rotation
```

The trailing x is kept so the algorithm still produces the same cube state. Use `remove_ending_rotations` to strip it when the final orientation does not matter.

**Multiple grips** are handled iteratively:

```
Input:   x  R  U  x  F  D    (two grips)
Pass 1:  R  F  x  D  B  x    (first x absorbed)
Pass 2:  R  F  B  U  x  x    (second x absorbed)
```

Key functions:
- `degrip_x_moves` / `degrip_y_moves` / `degrip_z_moves` — absorb single-axis rotations
- `degrip_full_moves` — absorb all rotation axes
- `has_grip(algo, config)` — detect if algorithm contains grip-breaking rotations

**Contrast with other transforms:**
- `translate_moves` applies a known, fixed orientation to the whole algorithm at once
- `translate_pov_moves` translates moves after inline rotations to the user's point of view without removing them
- `degrip` removes rotations by absorbing them into face moves

To absorb a y rotation, degrip calls `offset_yprime_moves` (the inverse offset) so subsequent moves land on the correct faces:

```
y  R  U  R' U'               (algorithm with grip)
↓  degrip absorbs y by calling offset_yprime_moves on suffix
B  U  B' U'  y               (degripped, trailing y preserved)
```

In [8]:
print('=== Degrip Transform ===')
print()

# Detect grip-breaking rotations
print('--- has_grip Detection ---')
test_algs = [
    "R U R' U'",
    "R x U R' U'",
    "R U y R' U'",
    "R U R' x",
]
for alg_str in test_algs:
    alg = Algorithm.parse_moves(alg_str)
    gripped, prefix, suffix, gripper = has_grip(alg, DEGRIP_FULL)
    print(f'  {alg!s:<18} has_grip={gripped}' +
          (f'  (gripper: {gripper})' if gripped else ''))

# Degrip x-axis rotations
print('\n--- Absorbing x Rotations ---')
alg = Algorithm.parse_moves("R x U R'")
degripped = alg.transform(degrip_x_moves)
print(f'  Original:  {alg}')
print(f'  Degripped: {degripped}')

# Verify same cube state
cube1 = VCube()
cube1.rotate(alg)
cube2 = VCube()
cube2.rotate(degripped)
print(f'  Same state? {cube1.is_equal(cube2, strict=False)}')

# Degrip all axes
print('\n--- Full Degrip Examples ---')
grip_examples = [
    "R U x F R'",
    "R y U R' U'",
    "F z R U R'",
    "R U R' F' R U R' U' R' F R2 U' R' y R U R' U'",
]
for alg_str in grip_examples:
    alg = Algorithm.parse_moves(alg_str)
    result = alg.transform(degrip_full_moves)

    # Verify
    c1 = VCube()
    c1.rotate(alg)
    c2 = VCube()
    c2.rotate(result)
    match = c1.is_equal(c2, strict=False)

    print(f'  {alg!s:<50} -> {result}  (verified: {match})')

# Practical example: CFOP algorithm with y rotation between steps
print('\n--- Practical: CFOP with Regrip ---')
cfop = Algorithm.parse_moves("R U R' U' y L' U' L")
print(f'  Original (with y regrip): {cfop}  ({cfop.metrics.htm} HTM, {cfop.metrics.rtm} rotations)')
smooth = cfop.transform(degrip_full_moves)
print(f'  Degripped:                {smooth}  ({smooth.metrics.htm} HTM, {smooth.metrics.rtm} rotations)')

=== Degrip Transform ===

--- has_grip Detection ---
  R U R' U'          has_grip=False
  R x U R' U'        has_grip=True  (gripper: x)
  R U y R' U'        has_grip=True  (gripper: y)
  R U R' x           has_grip=False

--- Absorbing x Rotations ---
  Original:  R x U R'
  Degripped: R F R' x
  Same state? True

--- Full Degrip Examples ---
  R U x F R'                                         -> R U D R' x  (verified: True)
  R y U R' U'                                        -> R U B' U' y  (verified: True)
  F z R U R'                                         -> F U L U' z  (verified: True)
  R U R' F' R U R' U' R' F R2 U' R' y R U R' U'      -> R U R' F' R U R' U' R' F R2 U' R' B U B' U' y  (verified: True)

--- Practical: CFOP with Regrip ---
  Original (with y regrip): R U R' U' y L' U' L  (7 HTM, 1 rotations)
  Degripped:                R U R' U' F' U' F y  (7 HTM, 1 rotations)


## Transform Deep Dive 4 — Symmetry (Mirror Across Slices)

**Module**: `cubing_algs.transform.symmetry`

Symmetry = reflection across a slice plane. Unlike offset (which rotates and preserves direction), symmetry **inverts** move direction — a clockwise R becomes a counter-clockwise L.

| Function | Plane | Effect |
|----------|-------|--------|
| `symmetry_m_moves` | M slice (between R and L) | R ↔ L with direction inversion |
| `symmetry_s_moves` | S slice (between F and B) | F ↔ B with direction inversion |
| `symmetry_e_moves` | E slice (between U and D) | U ↔ D with direction inversion |
| `symmetry_c_moves` | Combined M + S | Both R↔L and F↔B inversions |

The `SYMMETRY_TABLE` maps each symmetry type to `(ignore_set, mapping_dict)`. Moves in the ignore set (e.g., `x` and `M` for M-symmetry) pass through unchanged because they lie in the reflection plane.

In [9]:
print('=== Symmetry Transform ===')
print()

# Show SYMMETRY_TABLE for M
print("--- SYMMETRY_TABLE['M'] ---")
ignore_set, mapping = SYMMETRY_TABLE['M']
print(f'  Ignore set (in the mirror plane): {sorted(ignore_set)}')
print(f'  Mapping: {mapping}')

# Apply M-symmetry
print('\n--- M-Symmetry Examples ---')
alg = Algorithm.parse_moves("R U R' U'")
m_sym = alg.transform(symmetry_m_moves)
print(f"  {alg} -> {m_sym}  (R becomes L', U stays U', directions flip)")

alg2 = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")
print(f'  T-Perm: {alg2}')
print(f'  M-sym:  {alg2.transform(symmetry_m_moves)}')

# All four symmetries on the same algorithm
print('\n--- All Symmetries on Sexy Move ---')
base = Algorithm.parse_moves("R U R' U'")
print(f'  Original:   {base}')
print(f'  M-symmetry: {base.transform(symmetry_m_moves)}')
print(f'  S-symmetry: {base.transform(symmetry_s_moves)}')
print(f'  E-symmetry: {base.transform(symmetry_e_moves)}')
print(f'  C-symmetry: {base.transform(symmetry_c_moves)}')

# Symmetry vs Offset: the critical difference
print('\n--- Symmetry vs Offset ---')
alg = Algorithm.parse_moves("R U R' U'")
print(f'  Original:      {alg}')
print(f"  offset_y (y'): {alg.transform(offset_y_moves)}     (R->F, direction PRESERVED)")
print(f'  symmetry_s:    {alg.transform(symmetry_s_moves)}     (F->B, direction INVERTED)')
print()
print('  Offset = rotation (preserves chirality)')
print('  Symmetry = reflection (inverts chirality)')

# Verify symmetries produce valid cube states
print('\n--- Verification ---')
for name, sym_fn in [('M', symmetry_m_moves), ('S', symmetry_s_moves),
                      ('E', symmetry_e_moves), ('C', symmetry_c_moves)]:
    sym_alg = T_PERM.transform(sym_fn)
    cube = VCube()
    cube.rotate(sym_alg)
    cp, co, _, _, _ = cube.cubies
    is_pll = all(o == 0 for o in co)
    print(f'  {name}-sym T-Perm: {sym_alg}  (still PLL: {is_pll})')

=== Symmetry Transform ===

--- SYMMETRY_TABLE['M'] ---
  Ignore set (in the mirror plane): ['M', 'x']
  Mapping: {'F': 'F', 'S': 'S', 'z': 'z', 'U': 'U', 'y': 'y', 'R': 'L', 'x': 'x', 'B': 'B', 'L': 'R', 'M': 'M', 'D': 'D', 'E': 'E'}

--- M-Symmetry Examples ---
  R U R' U' -> L' U' L U  (R becomes L', U stays U', directions flip)
  T-Perm: R U R' F' R U R' U' R' F R2 U' R'
  M-sym:  L' U' L F L' U' L U L F' L2 U L

--- All Symmetries on Sexy Move ---
  Original:   R U R' U'
  M-symmetry: L' U' L U
  S-symmetry: R' U' R U
  E-symmetry: R' D' R D
  C-symmetry: L U L' U'

--- Symmetry vs Offset ---
  Original:      R U R' U'
  offset_y (y'): F U F' U'     (R->F, direction PRESERVED)
  symmetry_s:    R' U' R U     (F->B, direction INVERTED)

  Offset = rotation (preserves chirality)
  Symmetry = reflection (inverts chirality)

--- Verification ---
  M-sym T-Perm: L' U' L F L' U' L U L F' L2 U L  (still PLL: True)
  S-sym T-Perm: R' U' R B R' U' R U R B' R2 U R  (still PLL: True)
  E-sym 

## Transform Deep Dive 5 — Translate (POV Adaptation)

**Module**: `cubing_algs.transform.translate`

### `translate_moves` — Fixed Orientation Translation

`translate_moves` is a higher-order function: call it once with the orientation (a sequence of rotation moves), then apply the returned function to one or more algorithms. It rewrites all face moves so they produce the same effect on the reoriented cube.

Orientations use the two-letter notation from `ORIENTATIONS` (top-face + front-face). Standard is UF (white top, green front, no rotation). Each orientation maps to a rotation sequence (e.g. DF → z2, RF → z', FU → x).

**The problem this solves:**

A Bluetooth cube has no gyroscope — it only has mechanical sensors on each face. Those sensors always report moves in the cube's absolute frame (UF), no matter how the user holds the cube.

When the user picks an orientation before solving (e.g. DF), there is a mismatch between what the user does and what the cube records:

| What user sees | What the cube records (UF frame) |
|----------------|----------------------------------|
| R (right face) | L (it's physically the L face) |
| U (top face) | D (it's physically the D face) |
| R' | L' |
| U' | D' |

`translate_moves` bridges that gap:

```
translate_moves(z2):  L D L' D'  →  R U R' U'
                      (recorded)    (what user meant)
```

**More orientation examples:**

| Orientation | User does | Cube records | After translate |
|-------------|-----------|-------------|-----------------|
| DF (z2) | R U R' U' | L D L' D' | R U R' U' |
| RF (z') | R U R' U' | D R D' R' | R U R' U' |
| FU (x) | R U R' U' | R F R' F' | R U R' U' |
| FR (x y) | R U R' U' | U F U' F' | R U R' U' |

Also works for translating scrambles. A scramble generator produces moves in the standard UF frame. If the user holds the cube in DF, they need each move rewritten so they can apply it from their POV:

```
UF scramble:  R  U  F' D2 L  B' R2 U'
DF scramble:  L  D  F' U2 R  B' L2 D'
```

Both scrambles produce the exact same cube state — the user just reads different face names because they are holding the cube upside down.

### `translate_pov_moves` — Inline Rotation Translation

`translate_pov_moves` handles rotations discovered inline during the algorithm (gyroscope events). It walks the algorithm left-to-right, accumulates rotation effects, and translates non-rotation moves into the user's current point of view. Rotations are preserved in place.

**Typical use case:** a Bluetooth cube with a gyroscope (e.g. GAN iCarry) has two independent sensors:

- A **mechanical sensor** that detects face turns (R, U, F, ...). These are always reported relative to the cube's fixed physical stickers, regardless of how the cube is held.
- A **gyroscope** that detects whole-cube rotations (x, y, z). These are injected into the move stream as separate events.

Because these sensors are independent, the recorded algorithm mixes absolute-frame face moves with rotation events:

```
Sensor output:  R  U  R' U'  y  B  U  B' U'
                ^^^^^^^^^^^  ^  ^^^^^^^^^^^
                face sensor  gyro  face sensor
```

Here the user did y (rotated the cube) then continued solving what they see as the R face — but the cube still reports B because its stickers didn't move. `translate_pov_moves` rewrites the face moves after each rotation so the algorithm reads as the user intended:

```
After translate:  R  U  R' U'  y  R  U  R' U'
```

**Contrast with `translate_moves`:** `translate_moves` handles a known, fixed orientation applied to the whole algorithm at once. `translate_pov_moves` handles rotations discovered inline during the algorithm.

In [10]:
print('=== Translate Transform ===')
print()

# Create a translator for y orientation
y_orientation = Algorithm.parse_moves('y')
translator_y = translate_moves(y_orientation)

alg = Algorithm.parse_moves("R U R' U'")
translated = translator_y(alg)
print('--- Basic Translation ---')
print(f'  Original (from UF):  {alg}')
print(f'  Translated (for y):  {translated}  (as if cube was y-rotated)')

# Multiple rotation orientations
print('\n--- Translation for Different Orientations ---')
orientations = ['y', "y'", 'y2', 'x', 'x y']
for orient_str in orientations:
    orient = Algorithm.parse_moves(orient_str)
    t = translate_moves(orient)
    result = t(alg)
    print(f'  Orientation {orient_str:5s}: {alg} -> {result}')

# Verify: original at UF == translated at rotated orientation
print('\n--- Verification ---')
original_alg = Algorithm.parse_moves("R U R' U'")

cube1 = VCube()
cube1.rotate(original_alg)

# Translate for y, then apply from y orientation
translated_for_y = translate_moves(Algorithm.parse_moves('y'))(original_alg)
cube2 = VCube()
cube2.rotate('y')  # rotate to y orientation first
cube2.rotate(translated_for_y)

print(f'  Original at UF:        {cube1.state}')
print(f'  Translated after y:    {cube2.state}')
print(f'  Same puzzle?           {cube1.is_equal(cube2, strict=False)}')

# translate_pov_moves: handles embedded rotations
print('\n--- translate_pov_moves (Embedded Rotations) ---')
alg_with_rot = Algorithm.parse_moves("R U y R' U'")
pov_translated = alg_with_rot.transform(translate_pov_moves)
print(f'  Original:       {alg_with_rot}')
print(f'  POV translated: {pov_translated}')
print('  The y rotation stays, but subsequent moves are remapped')

# Practical: translate a PLL for back-slot execution
print('\n--- Practical: Algorithm for Different Slot ---')
t_perm = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")
back_translator = translate_moves(Algorithm.parse_moves('y2'))
t_perm_back = back_translator(t_perm)
print(f'  T-Perm (front): {t_perm}')
print(f'  T-Perm (back):  {t_perm_back}')

=== Translate Transform ===

--- Basic Translation ---
  Original (from UF):  R U R' U'
  Translated (for y):  F U F' U'  (as if cube was y-rotated)

--- Translation for Different Orientations ---
  Orientation y    : R U R' U' -> F U F' U'
  Orientation y'   : R U R' U' -> B U B' U'
  Orientation y2   : R U R' U' -> L U L' U'
  Orientation x    : R U R' U' -> R B R' B'
  Orientation x y  : R U R' U' -> F R F' R'

--- Verification ---
  Original at UF:        UULUUFUUFRRUBRRURRFFDFFUFFFDDRDDDDDDBLLLLLLLLBRRBBBBBB
  Translated after y:    UUUUUUFFLBRRBBBBBBRRUBRRURRRDDDDDDDDFFDFFUFFFBLLLLLLLL
  Same puzzle?           True

--- translate_pov_moves (Embedded Rotations) ---
  Original:       R U y R' U'
  POV translated: R U y F' U'
  The y rotation stays, but subsequent moves are remapped

--- Practical: Algorithm for Different Slot ---
  T-Perm (front): R U R' F' R U R' U' R' F R2 U' R'
  T-Perm (back):  L U L' B' L U L' U' L' B L2 U' L'


## Practical Applications — Putting It All Together

The five transform modules work together to solve real problems:

- **Canonical form**: strip rotations to compare algorithms by pure effect on pieces
- **Solver preparation**: orient to UF before solving
- **Algorithm libraries**: store in canonical orientation, translate on demand
- **Ergonomic optimization**: degrip → compress → choose best symmetry variant

In [11]:
print('=== Practical Pipeline ===')
print()

# Full pipeline: messy algorithm -> clean canonical form
messy = Algorithm.parse_moves("x y R U R' U' y' x' z R U R' z'")
print('--- Messy Algorithm Cleanup ---')
print(f'  Original: {messy}  ({messy.metrics.htm} HTM, {messy.metrics.rtm} rotations)')

step1 = messy.transform(degrip_full_moves)
print(f'  Degripped: {step1}')

step2 = step1.transform(compress_rotations)
print(f'  Compressed: {step2}')

step3 = step2.transform(remove_ending_rotations)
print(f'  No trailing rot: {step3}')

step4 = step3.transform(compress_moves)
final = step4
print(f'  Final: {final}  ({final.metrics.htm} HTM, {final.metrics.rtm} rotations)')

# Verify same effect
c1 = VCube()
c1.rotate(messy)
c2 = VCube()
c2.rotate(final)
print(f'  Same puzzle state? {c1.is_equal(c2, strict=False)}')

# Generate all symmetric variants and pick the shortest
print('\n--- Best Symmetric Variant ---')
base = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")
variants = {
    'Original': base,
    'M-symmetry': base.transform(symmetry_m_moves),
    'S-symmetry': base.transform(symmetry_s_moves),
    'E-symmetry': base.transform(symmetry_e_moves),
    'C-symmetry': base.transform(symmetry_c_moves),
}

for name, var in variants.items():
    print(f'  {name:<14}: {var}  ({var.metrics.htm} HTM)')

# Show algorithm in multiple orientations using translate
print('\n--- Algorithm in Multiple Orientations ---')
alg = Algorithm.parse_moves("R U R' U'")
for orient in ['y', "y'", 'y2', 'x']:
    t = translate_moves(Algorithm.parse_moves(orient))
    print(f'  From {orient:3s} orientation: {t(alg)}')

=== Practical Pipeline ===

--- Messy Algorithm Cleanup ---
  Original: x y R U R' U' y' x' z R U R' z'  (7 HTM, 6 rotations)
  Degripped: U F U' F' U L U' z' y x' x y' z
  Compressed: 
  No trailing rot: 
  Final:   (0 HTM, 0 rotations)
  Same puzzle state? False

--- Best Symmetric Variant ---
  Original      : R U R' F' R U R' U' R' F R2 U' R'  (13 HTM)
  M-symmetry    : L' U' L F L' U' L U L F' L2 U L  (13 HTM)
  S-symmetry    : R' U' R B R' U' R U R B' R2 U R  (13 HTM)
  E-symmetry    : R' D' R F R' D' R D R F' R2 D R  (13 HTM)
  C-symmetry    : L U L' B' L U L' U' L' B L2 U' L'  (13 HTM)

--- Algorithm in Multiple Orientations ---
  From y   orientation: F U F' U'
  From y'  orientation: B U B' U'
  From y2  orientation: L U L' U'
  From x   orientation: R B R' B'


## Wide Moves — The Bridge Between Face and Rotation

Wide moves occupy a unique position: they are **both** a face move and a rotation combined.

- `Rw` (wide R) = `L x` — turn two layers, equivalent to turning the opposite face plus a whole-cube rotation
- `Uw` (wide U) = `D y` — similarly for the U/D axis

Because wide moves combine both operations, they affect **both** `cp/co/ep/eo` (like a face move) **and** `so` (like a rotation) simultaneously. This dual nature is why the degrip transform can sometimes produce wide moves — absorbing a rotation doesn't always simplify to pure face moves.

The `wide` module provides:
- `unwide_rotation_moves` — expand wide moves into face + rotation (e.g., `Rw` → `L x`)
- `rewide_moves` — recombine face + rotation back into wide moves

In [12]:
print('=== Wide Moves ===')
print()

# Wide move decomposition at the cubie level
print('--- Cubie Impact Comparison ---')
solved = VCube()
s_cp, s_co, s_ep, s_eo, s_so = solved.cubies

for move_str, desc in [('R', 'face move'), ('x', 'rotation'), ('Rw', 'wide move')]:
    cube = VCube()
    cube.rotate(move_str)
    cp, co, ep, eo, so = cube.cubies
    cp_changed = cp != s_cp or co != s_co
    ep_changed = ep != s_ep or eo != s_eo
    so_changed = so != s_so
    print(f"  {move_str:3s} ({desc:10s}): pieces={'CHANGED' if cp_changed or ep_changed else 'unchanged':9s}  "
          f"orientation={'CHANGED' if so_changed else 'unchanged'}")

# Verify: Rw == L x
print('\n--- Wide = Opposite Face + Rotation ---')
cube_rw = VCube()
cube_rw.rotate('Rw')

cube_lx = VCube()
cube_lx.rotate('L x')

print(f'  Rw state: {cube_rw.state}')
print(f'  L x state: {cube_lx.state}')
print(f'  Identical? {cube_rw.state == cube_lx.state}')

# Expand and rewide
print('\n--- Expand / Rewide Transforms ---')
wide_alg = Algorithm.parse_moves("Rw U Rw'")
expanded = wide_alg.transform(unwide_rotation_moves)
print(f'  Wide:     {wide_alg}')
print(f'  Expanded: {expanded}')

# Rewide: combine face + rotation back to wide
face_rot = Algorithm.parse_moves("L x U L' x'")
rewided = face_rot.transform(rewide_moves)
print(f'\n  Face+rot: {face_rot}')
print(f'  Rewided:  {rewided}')

# Verify round-trip
c1 = VCube()
c1.rotate(wide_alg)
c2 = VCube()
c2.rotate(expanded)
print(f'\n  Wide == expanded? {c1.state == c2.state}')

=== Wide Moves ===

--- Cubie Impact Comparison ---
  R   (face move ): pieces=CHANGED    orientation=unchanged
  x   (rotation  ): pieces=unchanged  orientation=CHANGED
  Rw  (wide move ): pieces=CHANGED    orientation=CHANGED

--- Wide = Opposite Face + Rotation ---
  Rw state: UFFUFFUFFRRRRRRRRRFDDFDDFDDDBBDBBDBBLLLLLLLLLUUBUUBUUB
  L x state: UFFUFFUFFRRRRRRRRRFDDFDDFDDDBBDBBDBBLLLLLLLLLUUBUUBUUB
  Identical? True

--- Expand / Rewide Transforms ---
  Wide:     Rw U Rw'
  Expanded: L x U L' x'

  Face+rot: L x U L' x'
  Rewided:  r U r'

  Wide == expanded? True


## Combining Transforms — The Complete Rotation Toolkit

The five rotation-related modules form a complete toolkit. Here's when to reach for each:

| Goal | Module | Function |
|------|--------|----------|
| Rename moves for different viewpoint | **offset** | `offset_x_moves`, `rotate` |
| Simplify rotation sequences | **rotation** | `compress_rotations` |
| Remove mid-algorithm rotations | **degrip** | `degrip_full_moves` |
| Create mirror-image algorithm | **symmetry** | `symmetry_m_moves` |
| Adapt algorithm for different orientation | **translate** | `translate_moves` |
| Expand/contract wide moves | **wide** | `unwide_rotation_moves`, `rewide_moves` |

Transform composition with `algorithm.transform(*funcs, to_fixpoint=True)` chains them in order and can iterate until the result stabilizes.

In [13]:
print('=== Combined Transform Pipeline ===')
print()

# Start with a complex algorithm containing rotations and wide moves
complex_alg = Algorithm.parse_moves("x Rw U R' y U' Rw' x' z R U R' z'")
print(f'Original: {complex_alg}')
print(f'  HTM: {complex_alg.metrics.htm}, Rotations: {complex_alg.metrics.rtm}')

# Step 1: Expand wide moves to face + rotation
step1 = complex_alg.transform(unwide_rotation_moves)
print(f'\nStep 1 (expand wide):  {step1}')

# Step 2: Absorb rotations into face moves
step2 = step1.transform(degrip_full_moves)
print(f'Step 2 (degrip):       {step2}')

# Step 3: Compress rotation sequences
step3 = step2.transform(compress_rotations)
print(f'Step 3 (compress rot): {step3}')

# Step 4: Optimize moves
step4 = step3.transform(compress_moves)
print(f'Step 4 (compress):     {step4}')

# Step 5: Remove trailing rotations
step5 = step4.transform(remove_ending_rotations)
print(f'Step 5 (clean tail):   {step5}')

# Step 6: Rewide where possible
final = step5.transform(rewide_moves)
print(f'Step 6 (rewide):       {final}')
print(f'  HTM: {final.metrics.htm}, Rotations: {final.metrics.rtm}')

# Verify
c1 = VCube()
c1.rotate(complex_alg)
c2 = VCube()
c2.rotate(final)
print(f'\nSame puzzle state? {c1.is_equal(c2, strict=False)}')

# One-liner version using transform chain
print('\n--- Same Pipeline as One-Liner ---')
one_liner = complex_alg.transform(
    unwide_rotation_moves,
    degrip_full_moves,
    compress_rotations,
    compress_moves,
    remove_ending_rotations,
    rewide_moves,
)
print(f'  Result: {one_liner}')
print(f'  Matches step-by-step? {str(one_liner) == str(final)}')

=== Combined Transform Pipeline ===

Original: x Rw U R' y U' Rw' x' z R U R' z'
  HTM: 8, Rotations: 5

Step 1 (expand wide):  x L x U R' y U' L' x' x' z R U R' z'
Step 2 (degrip):       L D R' D' B' U B U' x y y z' y' y' x'
Step 3 (compress rot): y'
Step 4 (compress):     y'
Step 5 (clean tail):   
Step 6 (rewide):       
  HTM: 0, Rotations: 0

Same puzzle state? False

--- Same Pipeline as One-Liner ---
  Result: 
  Matches step-by-step? True


## Summary and Key Takeaways

### The Fundamental Duality
Face moves and rotations are fundamentally different operations that happen to look similar at the facelet level:
- **Face moves** change piece positions (`cp/co/ep/eo`) — they scramble the puzzle
- **Rotations** change spatial orientation (`so`) — they change your viewpoint

### The SO Array
The spatial orientation array is the hidden dimension of cube state. It tracks which original face occupies each position, enabling the library to distinguish between "the puzzle changed" and "the viewpoint changed".

### The 24 Orientations
A cube has exactly 24 valid orientations (6 top faces x 4 front faces). Two cubes with the same `cp/co/ep/eo` but different `so` represent the same puzzle from different viewpoints.

### The Five Transform Modules

| Module | What it does | When to use it |
|--------|-------------|----------------|
| **offset** | Renames face moves for a rotated viewpoint (pure remapping, no moves added or removed) | Converting algorithms between orientations; core building block for degrip and translate |
| **rotation** | Compresses rotation sequences via VCube simulation, strips rotations | Cleaning up redundant rotations; `compress_rotations` finds the shortest equivalent (≤2 moves) |
| **degrip** | Absorbs mid-algorithm rotations into face moves using inverse offset | Removing mid-algorithm regrips for speed; trailing rotations preserved for state equivalence |
| **symmetry** | Mirrors across slice planes (with direction inversion) | Generating algorithm variants |
| **translate** | Adapts algorithms for different POV orientations; handles both fixed orientations and inline gyroscope rotations | Bluetooth cube orientation mapping; applying algorithms from non-standard angles |

### Wide Moves: The Bridge
Wide moves (`Rw`, `Uw`) are the bridge between face moves and rotations — they affect both pieces and orientation simultaneously. They can be expanded to face + rotation (`unwide_rotation_moves`) and recombined (`rewide_moves`).

In [14]:
print('=== Final Demo: The Core Concepts ===')
print()

# The fundamental duality in one example
print('--- The Duality ---')
cube = VCube()
cube.rotate('R')  # face move
_, _, _, _, so_after_r = cube.cubies

cube2 = VCube()
cube2.rotate('x')  # rotation
cp_after_x, co_after_x, ep_after_x, eo_after_x, _ = cube2.cubies

print(f'  R changes pieces, not orientation: SO={so_after_r} (still solved)')
print(f'  x changes orientation, not pieces:  CP={cp_after_x} CO={co_after_x} (still identity)')

# Quick reference of all transforms
print('\n--- Transform Quick Reference ---')
alg = Algorithm.parse_moves("R U R' U'")
print(f'  Source algorithm:    {alg}')
print(f'  offset_x:           {alg.transform(offset_x_moves)}')
print(f'  symmetry_m:         {alg.transform(symmetry_m_moves)}')
print(f"  translate(y):       {translate_moves(Algorithm.parse_moves('y'))(alg)}")

messy = Algorithm.parse_moves("R x U R' x'")
print(f'\n  With rotations:     {messy}')
print(f'  degrip_full:        {messy.transform(degrip_full_moves)}')
print(f'  compress_rotations: {messy.transform(compress_rotations)}')
print(f'  remove_rotations:   {messy.transform(remove_rotations)}')

print('\n' + '=' * 55)
print('Face moves change pieces. Rotations change viewpoint.')
print('Same puzzle, different perspective.')
print('=' * 55)

=== Final Demo: The Core Concepts ===

--- The Duality ---
  R changes pieces, not orientation: SO=[0, 1, 2, 3, 4, 5] (still solved)
  x changes orientation, not pieces:  CP=[0, 1, 2, 3, 4, 5, 6, 7] CO=[0, 0, 0, 0, 0, 0, 0, 0] (still identity)

--- Transform Quick Reference ---
  Source algorithm:    R U R' U'
  offset_x:           R B R' B'
  symmetry_m:         L' U' L U
  translate(y):       F U F' U'

  With rotations:     R x U R' x'
  degrip_full:        R F R' x' x
  compress_rotations: 
  remove_rotations:   R U R'

Face moves change pieces. Rotations change viewpoint.
Same puzzle, different perspective.
